# PHASE 5 — ADVANCED WORKFLOWS & PRODUCTION


# Day 29 — Model Production (Joblib)


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Explain what it means to put a model into "Production".
- Use `joblib` to serialize (save) a trained Scikit-learn Pipeline to your hard drive.
- Deserialize (load) that model into a brand new Python script to make predictions.
- Understand why saving the full Pipeline (and not just the model) is absolutely critical for production.


## 2. Prerequisites
- Day 5 (Pipelines).


## 3. Concept: The Jupyter Notebook Trap
For the last 28 days, every time you ran a notebook, it spent 10 seconds doing `.fit()` to train the model, and then immediately made predictions. 

In the real world, you do not train a model every time a customer clicks a button on a website. Training on massive datasets can take days and cost thousands of dollars in cloud computing. 

Instead, you train the model **once**. You save the "brain" of the model to a file (like saving a Word document). Then, a web developer loads that tiny file onto a web server. When a customer clicks a button, the server just calls `.predict()` (which takes 0.001 seconds).


## 4. Concept: Serialization (Pickling)
In Python, saving an object (like a trained Scikit-learn Pipeline) to the hard drive is called **Serialization** (or "Pickling"). 
It converts the mathematical arrays, the trees, and the scalers into a byte-stream and writes it to a file, usually with a `.pkl` or `.joblib` extension.


## 5. Scikit-learn API
```python
import joblib
# Save to hard drive
joblib.dump(trained_pipeline, 'my_model.joblib')

# Load from hard drive (in a totally different script!)
loaded_model = joblib.load('my_model.joblib')
```


## 6. Simple Example: Saving a Model
Let's train a Pipeline on some data, just like we always do, and then save it to the hard drive.


In [ ]:
import pandas as pd
import joblib
import os
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Generate Training Data
X_train, y_train = make_classification(n_samples=1000, n_features=5, random_state=42)

# 2. Build and Train the Pipeline
production_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42))
])

print('Training model...')
production_pipe.fit(X_train, y_train)
print('Training complete!')

# 3. Save the Pipeline to the hard drive
joblib.dump(production_pipe, 'spam_detector_v1.joblib')

print(f'\nModel saved successfully! File size: {os.path.getsize("spam_detector_v1.joblib") / 1024:.1f} KB')


## 7. Code Walkthrough
- We ran `.fit()` on the Pipeline.
- The `StandardScaler` calculated the means and standard deviations of the 1,000 rows.
- The `RandomForestClassifier` built 50 decision trees.
- `joblib.dump()` took all of that math and saved it to a file named `spam_detector_v1.joblib`.


## 8. Experiment: Loading the Model (Production Simulation)
Imagine this next cell is a completely different Python script running on a web server in another country. 
A user just filled out a form on a website, and the server received 1 row of data. 
The server does NOT have the training data. It does NOT call `.fit()`. It simply loads the `.joblib` file and calls `.predict()`!


In [ ]:
# Assume this is a new script (e.g., app.py on a Flask/Django server)

# 1. Load the model from the hard drive
server_model = joblib.load('spam_detector_v1.joblib')
print('Model loaded from disk successfully!')

# 2. User submits 1 row of new data from the website
user_input = pd.DataFrame([[0.5, -1.2, 3.4, 0.1, -0.9]])

# 3. Make the prediction instantly
prediction = server_model.predict(user_input)
probability = server_model.predict_proba(user_input)[0][1]

print(f'\nPrediction: Class {prediction[0]}')
print(f'Confidence: {probability * 100:.1f}%')


> Notice how incredibly fast that was! The server didn't have to train anything. It just executed the pre-learned math.


## 9. Prediction Exercise
Read the following scenario, but **DO NOT RUN IT YET**.


In [ ]:
# Bad Practice: Saving just the model, not the pipeline
bad_scaler = StandardScaler()
X_scaled = bad_scaler.fit_transform(X_train)

bad_rf = RandomForestClassifier()
bad_rf.fit(X_scaled, y_train)

joblib.dump(bad_rf, 'bad_model.joblib')


> **Question:** The engineer above scaled the data manually, trained the Random Forest, and saved *only* the Random Forest to the hard drive. 
> When the web server loads `bad_model.joblib` and receives raw `user_input`, what will happen when it calls `predict()`?

**Think before running the next cell!**


In [ ]:
print('Error: The model will output complete garbage!')
print('Why? The Random Forest was trained on SCALED data (values between -3 and 3).')
print('If the user inputs a raw Salary of $85,000, the Random Forest has no idea what to do with that massive number.')
print('Because the engineer didn\'t save the Scaler, the web server has no way to scale the $85,000 down to the correct proportion.')


## 10. The Golden Rule of Production
**ALWAYS SAVE THE PIPELINE.**
If you save the entire Pipeline to the `.joblib` file, the Pipeline remembers the `StandardScaler`. It remembers the exact Means and Standard Deviations from the training set. 
When the web server calls `server_model.predict(user_input)`, the Pipeline automatically routes the raw `$85,000` through the saved Scaler, transforms it to `1.2`, and then passes `1.2` to the Random Forest. It guarantees the math is perfectly identical to the training environment!


## 11. Coding Exercise
Load the model `spam_detector_v1.joblib` one more time. 
We can actually peek inside the loaded file to prove it saved the Scaler's math.
Print out `server_model.named_steps['scaler'].mean_`. This will output the 5 means the scaler memorized during training!


In [ ]:
# YOUR CODE HERE
server_model = joblib.load('spam_detector_v1.joblib')
saved_means = server_model.named_steps['scaler'].mean_
print('The means memorized by the saved Scaler:')
print(saved_means)


## 12. Debugging Challenge
A company trains a model using Scikit-learn `v1.5.0` on a Windows machine. They save it as `model.joblib`. 
They send it to their DevOps team, who loads it onto a Linux server running Scikit-learn `v0.22.0`. The server crashes with a `ModuleNotFoundError` or `ValueError` when trying to load the file. Why?


In [ ]:
# Conceptual Bug
print('Error: Joblib files are highly version-dependent.')
print('A Pipeline saved in Scikit-learn v1.5 cannot be loaded by Scikit-learn v0.22.')
print('The internal Python code for the Random Forest changed between those versions, so the byte-stream no longer makes sense to the older version.')


> **Rule:** Your Training Environment (Jupyter) and your Production Environment (Web Server) MUST use the exact same version of Scikit-learn and Python! (This is usually managed via `requirements.txt` or Docker).


## 13. Model Evaluation (Monitoring)
Once a model is in production, your job isn't over. Models suffer from **Data Drift**. 
If you trained a house price model in 2019, and deploy it in 2024, it will massively underpredict prices because of inflation. 
You must constantly monitor the model's predictions in production, and if they start becoming inaccurate, you must pull the newest data, retrain a new model, and save a `v2.joblib` file!


## 14. Real-World Example
**Zillow Zestimate**: Zillow's data scientists train massive gradient boosting pipelines on historical housing data. They save these pipelines as `.pkl` or `.joblib` files and deploy them to cloud servers (AWS/GCP). When you open the Zillow app and look at a house, your phone sends the house's features (Beds, Baths, SqFt) to the AWS server. The server loads the `.joblib` file into RAM, runs `.predict()`, and sends the price back to your phone in milliseconds.


## 15. Mini Project
Let's clean up our hard drive. Use Python's built-in `os` module to delete the `spam_detector_v1.joblib` file we created earlier, just to prove we know how to manage files!


In [ ]:
import os

file_path = 'spam_detector_v1.joblib'
if os.path.exists(file_path):
    os.remove(file_path)
    print(f'{file_path} has been successfully deleted from the hard drive.')
else:
    print('File not found.')


## 16. Common Mistakes
- **Not saving the Scaler**: The most catastrophic mistake a Junior Data Scientist makes. The web server must have the exact same Scaler object that was used during training. Put it in a Pipeline!
- **Version Mismatches**: Training on Python 3.12 and deploying on Python 3.8. It will crash.
- **Uploading massive datasets to the server**: The web server does not need `X_train.csv`. It only needs the 500KB `.joblib` file!


## 17. Interview Questions
- **Beginner**: What library do we use to save a Scikit-learn model to the hard drive? (Answer: `joblib` or `pickle`).
- **Intermediate**: Why is it critical to save a `Pipeline` instead of just the final algorithm? (Answer: Because the preprocessing steps, like `StandardScaler` or `OneHotEncoder`, contain learned math (like means or categories). If you don't save them, the production server cannot process raw user input correctly).
- **Advanced**: What is Data Drift? (Answer: The phenomenon where a deployed model's accuracy degrades over time because the real-world data it is predicting on has fundamentally shifted away from the historical data it was trained on).


## 18. Knowledge Check
- What function saves the model? (`joblib.dump()`)
- What function loads the model? (`joblib.load()`)
- Does `.load()` require you to run `.fit()` again? (No, the model is already trained)


## 19. Summary
- **Production** means exposing your trained model to the real world (usually via a web server).
- **Joblib** is used to save (`dump`) and load (`load`) models.
- You must ALWAYS save the full **Pipeline** so the server remembers how to scale and encode raw user data.
- Web servers only call `.predict()`, never `.fit()`.
- You must ensure Python and Scikit-learn versions match between training and production environments.


## 20. Homework
Tomorrow is Day 30, the Final Capstone! 
Take today to review Phase 4 and 5. Make sure you fully understand Pipelines, Cross-Validation, and GridSearchCV, as you will need all of them to pass the final exam!
